# Grouping and aggregation

## Learning objectives

By the end of this notebook you will be able to:

- explain split-apply-combine
- summarise by one or more groups with groupby
- compute several statistics at once with agg and named aggregation
- add group statistics to every row with transform
- build frequency and summary tables with crosstab and pivot_table

## Introduction

"What is the mean glucose by group?", "How many positives per site and month?" Group summaries answer
most descriptive questions. pandas handles them with `groupby`: **split** the rows into groups,
**apply** a calculation to each, **combine** the results into a table.

In [1]:
import pandas as pd
import numpy as np
RAW = "../datasets/raw/"
PROCESSED = "../datasets/processed/"

In [2]:
patients = pd.read_csv(RAW + "patients.csv")
labs = pd.read_csv(RAW + "lab_results.csv")
clean = pd.read_csv(PROCESSED + "sample_register_clean.csv", parse_dates=["collected"])

`sample_register_clean.csv` is written by notebook 05. Run that notebook first if this cell fails.

## One group, one statistic

In [3]:
patients.groupby("site")["age"].mean().round(1)

site
Site A    49.6
Site B    56.0
Site C    50.8
Site D    48.2
Name: age, dtype: float64

In [4]:
patients.groupby("site").size()          # rows per group

site
Site A    111
Site B     87
Site C     54
Site D     48
dtype: int64

## Several groups

In [5]:
patients.groupby(["site", "sex"])["age"].mean().round(1)

site    sex
Site A  F      45.3
        M      54.0
Site B  F      54.3
        M      58.0
Site C  F      49.3
        M      52.1
Site D  F      45.0
        M      51.3
Name: age, dtype: float64

The result has a two-level index. `unstack` spreads the inner level into columns, which is often
easier to read:

In [6]:
patients.groupby(["site", "sex"])["age"].mean().round(1).unstack()

sex,F,M
site,,
Site A,45.3,54.0
Site B,54.3,58.0
Site C,49.3,52.1
Site D,45.0,51.3


## Several statistics: agg

In [7]:
patients.groupby("group")["age"].agg(["count", "mean", "median", "std", "min", "max"]).round(1)

,count,mean,median,std,min,max
group,,,,,,
Control,158,52.2,53.0,20.8,18,85
Treatment,142,50.6,49.0,19.4,19,85


**Named aggregation** chooses the column, the function and the output name together. This is the
clearest way to build a summary table:

In [8]:
summary = patients.groupby("site").agg(
    n=("patient_id", "count"),
    mean_age=("age", "mean"),
    sd_age=("age", "std"),
    pct_female=("sex", lambda s: (s == "F").mean() * 100),
    weight_missing=("weight_kg", lambda s: s.isna().sum()),
).round(1)
summary

,n,mean_age,sd_age,pct_female,weight_missing
site,,,,,
Site A,111,49.6,20.8,50.5,4
Site B,87,56.0,18.6,54.0,4
Site C,54,50.8,20.8,48.1,0
Site D,48,48.2,19.4,50.0,2


Reading the table: one row per site, with the number of patients, their mean and standard deviation of
age, the percentage who are female, and how many weights are missing. Compare sites on these before
comparing outcomes, because differences in who attends a site can explain differences in results.

## Positivity by group

The mean of a True/False column is the proportion of True values:

In [9]:
tested = clean.dropna(subset=["result"])
by_type = tested.groupby("sample_type", observed=True).agg(
    tests=("result", "size"),
    positives=("result", lambda r: (r == "Positive").sum()),
)
by_type["positivity_pct"] = (by_type["positives"] / by_type["tests"] * 100).round(1)
by_type

,tests,positives,positivity_pct
sample_type,,,
Blood,201,57,28.4
Saliva,69,22,31.9
Urine,130,39,30.0


`observed=True` limits the result to categories that occur, which is the recommended setting for
categorical columns.

## Grouping by time

In [10]:
monthly = (tested
           .groupby(tested["collected"].dt.to_period("M"))
           .agg(tests=("result", "size"), positive_pct=("result", lambda r: (r == "Positive").mean() * 100))
           .round(1))
monthly.head(6)

,tests,positive_pct
collected,,
2025-01,33,12.1
2025-02,33,30.3
2025-03,41,34.1
2025-04,34,35.3
2025-05,27,29.6
2025-06,35,34.3


## transform: group values on every row

`agg` gives one row per group. `transform` returns a value **for every original row**, which is how you
compare each row to its group.

In [11]:
labs["patient_mean_glucose"] = labs.groupby("patient_id")["glucose_mmol"].transform("mean")
labs["diff_from_own_mean"] = (labs["glucose_mmol"] - labs["patient_mean_glucose"]).round(2)
labs[["patient_id", "visit", "glucose_mmol", "patient_mean_glucose", "diff_from_own_mean"]].head(6).round(2)

,patient_id,visit,glucose_mmol,patient_mean_glucose,diff_from_own_mean
0,P0001,0,4.8,4.75,0.05
1,P0001,1,4.7,4.75,-0.05
2,P0002,0,5.6,5.23,0.37
3,P0002,1,4.8,5.23,-0.43
4,P0002,2,5.3,5.23,0.07
5,P0003,0,6.7,6.20,0.50


Standardising within groups (z-scores per site) is a typical use:

In [12]:
p = patients.copy()
g = p.groupby("site")["age"]
p["age_z_within_site"] = ((p["age"] - g.transform("mean")) / g.transform("std")).round(2)
p[["site", "age", "age_z_within_site"]].head()

,site,age,age_z_within_site
0,Site A,41,-0.41
1,Site B,83,1.45
2,Site B,36,-1.07
3,Site A,24,-1.23
4,Site A,77,1.32


## filter: keep whole groups

Keep only patients with three or more lab visits:

In [13]:
three_plus = labs.groupby("patient_id").filter(lambda g: len(g) >= 3)
print(labs["patient_id"].nunique(), "patients;", three_plus["patient_id"].nunique(), "with 3+ visits")

300 patients; 193 with 3+ visits


## First, last and change per group

In [14]:
bp = (labs.sort_values(["patient_id", "visit"])
      .groupby("patient_id")["systolic_bp"]
      .agg(first="first", last="last", n="size"))
bp["change"] = bp["last"] - bp["first"]
bp[bp["n"] > 1]["change"].describe().round(1)

count    300.0
mean       0.2
std       22.0
min      -61.0
25%      -14.0
50%       -0.5
75%       16.0
max       66.0
Name: change, dtype: float64

## crosstab: frequency tables

In [15]:
pd.crosstab(patients["site"], patients["smoker"], margins=True)

smoker,No,Yes,All
site,,,
Site A,85,26,111
Site B,67,20,87
Site C,42,12,54
Site D,37,11,48
All,231,69,300


In [16]:
pd.crosstab(patients["site"], patients["smoker"], normalize="index").round(3) * 100

smoker,No,Yes
site,,
Site A,76.6,23.4
Site B,77.0,23.0
Site C,77.8,22.2
Site D,77.1,22.9


`normalize="index"` gives row percentages: within each site, the percentage of smokers and non-smokers.

## pivot_table: summaries in a grid

In [17]:
merged = labs.merge(patients[["patient_id", "group"]], on="patient_id")
pd.pivot_table(merged, values="systolic_bp", index="visit", columns="group", aggfunc="mean").round(1)

group,Control,Treatment
visit,,
0,127.9,127.3
1,126.2,128.9
2,125.5,127.2
3,128.0,129.5


`merge` joins the two tables (next notebook). The table shows mean systolic blood pressure at each visit
for each study group, a typical way to present follow-up data.

## Practise

Exercises for this folder are in `exercises/Pandas_Exercises.ipynb`.

## Further reading

- [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Reshaping and pivot tables](https://pandas.pydata.org/docs/user_guide/reshaping.html)